In [1]:
# Install dependencies
!pip install \
    datasets \
    sentence-transformers \
    faiss-cpu \
    ollama \
    ragas \
    numpy

In [4]:
import json
import os
import time
import random
import traceback
from datetime import datetime

import numpy as np
import faiss
import ollama
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from ragas import evaluate
from ragas.metrics import context_recall, context_precision, faithfulness

print("All imports OK")

All imports OK


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_27416/363705079.py:14: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_recall, context_precision, faithfulness
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_27416/363705079.py:14: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_recall, context_precision, faithfulness
/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_27416/363705079.py:14: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Exa

datasets — load QASPER from HuggingFace
sentence_transformers — embed text with all-MiniLM-L6-v2
faiss — vector index for retrieval
ollama — call Llama 3.2 3B locally
ragas — compute Context Recall, Context Precision, Faithfulness
json, os, time, random — file handling and timing

In [7]:
# ── LOCKED CONFIG — do not change between V0, V1, V2

# Chunking — V0 specific, will change in V1/V2
CHUNK_SIZE    = 256   # tokens
CHUNK_OVERLAP = 32    # tokens — 12.5% overlap

# Retrieval — fixed across all variants
TOP_K         = 5

# Models — fixed across all variants
EMBED_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL     = "llama3.2:3b"
TEMPERATURE   = 0     # greedy decoding — reproducibility

# Sampling
N_PILOT       = 10
RANDOM_SEED   = 42

# Paths
DATA_DIR      = "data"
os.makedirs(DATA_DIR, exist_ok=True)

QUERIES_FILE  = os.path.join(DATA_DIR, "queries.json")   # shared V0/V1/V2
CHUNKS_FILE   = os.path.join(DATA_DIR, "v0_chunks.json")
INDEX_FILE    = os.path.join(DATA_DIR, "v0_index.faiss")
RESULTS_FILE  = os.path.join(DATA_DIR, "v0_results.json")

# Prompt — identical across V0, V1, V2
SYSTEM_PROMPT = (
    "You are a research assistant. Answer the question using ONLY the "
    "provided context. If the context does not contain enough information, "
    "say: I cannot answer this question from the provided context. "
    "Be concise and precise."
)

print("Config set.")
print(f"  Chunk size:   {CHUNK_SIZE} tokens, {CHUNK_OVERLAP} overlap")
print(f"  Top-K:        {TOP_K}")
print(f"  Embed model:  {EMBED_MODEL}")
print(f"  LLM:          {LLM_MODEL}  temperature={TEMPERATURE}")
print(f"  Pilot:        {N_PILOT} queries")

Config set.
  Chunk size:   256 tokens, 32 overlap
  Top-K:        5
  Embed model:  sentence-transformers/all-MiniLM-L6-v2
  LLM:          llama3.2:3b  temperature=0
  Pilot:        10 queries


In [11]:
print("Loading QASPER...")
ds = load_dataset(
    "allenai/qasper",
    revision="refs/convert/parquet",
    trust_remote_code=False,
)
papers = ds["train"]
print(f"Papers loaded: {len(papers)}")
print(f"Splits available: {list(ds.keys())}")

Loading QASPER...
Papers loaded: 888
Splits available: ['train', 'validation', 'test']


### Inspect structure of one paper. Do not skip this. Every accessor we write in the next cells depends on knowing exactly what the fields look like.

In [15]:
paper = papers[0]

print("TOP-LEVEL KEYS:", list(paper.keys()))
print()

# Inspect full_text
full_text = paper.get("full_text", {})
print("FULL_TEXT KEYS:", list(full_text.keys()))
section_names   = full_text.get("section_name", [])
paragraph_lists = full_text.get("paragraphs", [])
print(f"  section_name:  {len(section_names)} sections")
print(f"  paragraphs:    {len(paragraph_lists)} paragraph lists")
print(f"  First section: '{section_names[0]}'")
print(f"  First para:    '{paragraph_lists[0][0][:150]}'")
print()

# Inspect qas — Parquet makes this a dict of lists, not list of dicts
qas = paper.get("qas", {})
print("QAS TYPE:", type(qas))
print("QAS KEYS:", list(qas.keys()))
n_questions = len(qas.get("question", []))
print(f"  Questions in this paper: {n_questions}")
print(f"  First question: '{qas['question'][0]}'")
print()

# Inspect one answer
answers_wrapper = qas["answers"][0]
print("ANSWERS WRAPPER KEYS:", list(answers_wrapper.keys()))
answer = answers_wrapper["answer"][0]
print("ANSWER KEYS:", list(answer.keys()))
print(f"  free_form_answer: '{answer.get('free_form_answer', '')[:100]}'")
print(f"  extractive_spans: {answer.get('extractive_spans', [])}")
print(f"  evidence:         '{answer.get('evidence', [''])[0][:100]}'")
print(f"  yes_no:           {answer.get('yes_no')}")
print(f"  unanswerable:     {answer.get('unanswerable')}")

TOP-LEVEL KEYS: ['id', 'title', 'abstract', 'full_text', 'qas', 'figures_and_tables']

FULL_TEXT KEYS: ['section_name', 'paragraphs']
  section_name:  21 sections
  paragraphs:    21 paragraph lists
  First section: 'Introduction'
  First para:    'Affective events BIBREF0 are events that typically affect people in positive or negative ways. For example, getting money and playing sports are usual'

QAS TYPE: <class 'dict'>
QAS KEYS: ['question', 'question_id', 'nlp_background', 'topic_background', 'paper_read', 'search_query', 'question_writer', 'answers']
  Questions in this paper: 9
  First question: 'What is the seed lexicon?'

ANSWERS WRAPPER KEYS: ['answer', 'annotation_id', 'worker_id']
ANSWER KEYS: ['unanswerable', 'extractive_spans', 'yes_no', 'free_form_answer', 'evidence', 'highlighted_evidence']
  free_form_answer: 'a vocabulary of positive and negative predicates that helps determine the polarity score of an event'
  extractive_spans: []
  evidence:         'The seed lexico

#### map what we just learned before moving on:

+ Text lives in full_text["section_name"] and full_text["paragraphs"]
+ QAS is columnar — qas["question"][i], qas["answers"][i]
+ Answer fields are free_form_answer, extractive_spans, evidence, yes_no, unanswerable
+ No answer_string — that was the original script format, Parquet renamed it to free_form_answer

#### now write it for one paper first, verify the output, then scale to all 888.

In [22]:
def get_answer_type(answer: dict) -> str:
    """
    Classify a single QASPER answer into one of four types.
    Priority order matters — check unanswerable first.
    """
    if answer.get("unanswerable", False):
        return "unanswerable"
    if answer.get("yes_no") is not None:
        return "yes_no"
    if answer.get("extractive_spans"):
        return "extractive"
    if answer.get("free_form_answer", "").strip():
        return "abstractive"
    return "unanswerable"


def extract_queries_from_paper(paper_id: str, paper: dict) -> list:
    """
    Extract all answerable QA pairs from one QASPER paper.
    
    Parquet structure: qas is a dict of lists.
    qas["question"][i] and qas["answers"][i] are paired by index.
    """
    queries = []
    qas = paper.get("qas", {})
    if not isinstance(qas, dict):
        return queries

    questions    = qas.get("question", [])
    answers_list = qas.get("answers", [])

    for i, question in enumerate(questions):
        question = question.strip()
        if not question:
            continue

        answers_wrapper = answers_list[i] if i < len(answers_list) else {}
        answers = answers_wrapper.get("answer", []) if isinstance(answers_wrapper, dict) else []
        if not answers:
            continue

        answer      = answers[0]   # first annotator only
        answer_type = get_answer_type(answer)

        if answer_type == "unanswerable":
            continue

        # Build ground truth string based on type
        yes_no    = answer.get("yes_no")
        free_form = answer.get("free_form_answer", "").strip()
        ext_spans = answer.get("extractive_spans", [])
        evidence  = answer.get("evidence", [])

        if answer_type == "yes_no":
            answer_string = "Yes" if yes_no else "No"
        elif ext_spans:
            answer_string = " ".join(ext_spans).strip()
        else:
            answer_string = free_form

        if not answer_string:
            continue

        queries.append({
            "paper_id":    paper_id,
            "question":    question,
            "answer":      answer_string,
            "evidence":    evidence,
            "answer_type": answer_type,
        })

    return queries


# Test on first paper before scaling
test_queries = extract_queries_from_paper(papers[0]["id"], papers[0])
print(f"Queries extracted from paper 0: {len(test_queries)}")
print()
for q in test_queries[:3]:
    print(f"  type:     {q['answer_type']}")
    print(f"  question: {q['question'][:80]}")
    print(f"  answer:   {q['answer'][:80]}")
    print(f"  evidence: {q['evidence'][0][:80] if q['evidence'] else 'none'}")
    print()

Queries extracted from paper 0: 9

  type:     abstractive
  question: What is the seed lexicon?
  answer:   a vocabulary of positive and negative predicates that helps determine the polari
  evidence: The seed lexicon consists of positive and negative predicates. If the predicate 

  type:     abstractive
  question: What are the results?
  answer:   Using all data to train: AL -- BiGRU achieved 0.843 accuracy, AL -- BERT achieve
  evidence: FLOAT SELECTED: Table 3: Performance of various models on the ACP test set.

  type:     abstractive
  question: How are relations used to propagate polarity?
  answer:   based on the relation between events, the suggested polarity of one event can de
  evidence: In this paper, we propose a simple and effective method for learning affective e



#### queries extracted, types correct, evidence present

In [27]:
print("Extracting queries from all 888 papers...")
t0 = time.time()

all_queries = []
for paper in papers:
    paper_id = paper.get("id", "unknown")
    all_queries.extend(extract_queries_from_paper(paper_id, paper))

elapsed = time.time() - t0

# Count by type
type_counts = {}
for q in all_queries:
    t = q["answer_type"]
    type_counts[t] = type_counts.get(t, 0) + 1

print(f"Done in {elapsed:.1f}s")
print(f"Total answerable queries: {len(all_queries)}")
print()
print("Breakdown by type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t:12}: {c:5}  ({100*c/len(all_queries):.1f}%)")

Extracting queries from all 888 papers...
Done in 0.2s
Total answerable queries: 2321

Breakdown by type:
  abstractive :   610  (26.3%)
  extractive  :  1314  (56.6%)
  yes_no      :   397  (17.1%)


#### Abstractive questions are harder for RAG because the answer requires synthesis across evidence, not just span extraction. Oversampling them stresses the pipeline more.

In [30]:
def stratified_sample(all_queries: list, n: int, seed: int) -> list:
    """
    Sample n queries with target proportions:
      50% extractive, 30% abstractive, 20% yes_no
    If a bucket has fewer than the target, remainder fills
    from the largest available bucket.
    """
    rng = random.Random(seed)

    buckets = {"extractive": [], "abstractive": [], "yes_no": []}
    for q in all_queries:
        t = q["answer_type"]
        if t in buckets:
            buckets[t].append(q)

    for bucket in buckets.values():
        rng.shuffle(bucket)

    targets = {
        "extractive":  int(n * 0.50),
        "abstractive": int(n * 0.30),
        "yes_no":      int(n * 0.20),
    }

    sampled = []
    used    = set()

    for t, target in targets.items():
        take = min(target, len(buckets[t]))
        for q in buckets[t][:take]:
            sampled.append(q)
            used.add(q["question"])

    # Fill any remainder from largest bucket
    for bucket in sorted(buckets.values(), key=len, reverse=True):
        for q in bucket:
            if len(sampled) >= n:
                break
            if q["question"] not in used:
                sampled.append(q)
                used.add(q["question"])

    sampled = sampled[:n]
    rng.shuffle(sampled)
    return sampled


# Sample and save
queries = stratified_sample(all_queries, N_PILOT, RANDOM_SEED)

with open(QUERIES_FILE, "w") as f:
    json.dump(queries, f, indent=2)

# Verify composition
sc = {}
for q in queries:
    sc[q["answer_type"]] = sc.get(q["answer_type"], 0) + 1

print(f"Saved {len(queries)} queries → {QUERIES_FILE}")
print()
print("Sample composition:")
for t, c in sorted(sc.items()):
    print(f"  {t}: {c}")
print()
print("First query:")
q0 = queries[0]
print(f"  paper_id:    {q0['paper_id']}")
print(f"  question:    {q0['question'][:80]}")
print(f"  answer:      {q0['answer'][:80]}")
print(f"  answer_type: {q0['answer_type']}")
print(f"  evidence:    {q0['evidence'][0][:80] if q0['evidence'] else 'none'}")

Saved 10 queries → data/queries.json

Sample composition:
  abstractive: 3
  extractive: 5
  yes_no: 2

First query:
  paper_id:    1811.11365
  question:    Why is this work different from text-only UNMT?
  answer:      the image can play the role of a pivot “language" to bridge the two languages wi
  answer_type: extractive
  evidence:    Our idea is originally inspired by the text-only unsupervised MT (UMT) BIBREF8 ,


### Load embedding model.
We load this now because we need its tokenizer for chunking. The tokenizer is what counts tokens accurately — without it, "256 tokens" is just an approximation based on whitespace.

In [33]:
print(f"Loading embedding model: {EMBED_MODEL}")
t0 = time.time()

embed_model = SentenceTransformer(EMBED_MODEL)
tokenizer   = embed_model.tokenizer

print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Embedding dimension: {embed_model.get_sentence_embedding_dimension()}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded in 2.9s
Embedding dimension: 384


### Fixed window chunking function.

In [36]:
def chunk_fixed_window(text: str, tokenizer, chunk_size: int, overlap: int) -> list:
    """
    Split text into fixed-size token windows with overlap.

    V0 design decisions:
    - Uses the model tokenizer to count tokens accurately
    - No awareness of sentence, paragraph or section boundaries
    - A concept spanning a paragraph boundary gets split arbitrarily
    - This structural blindness is intentional — it is what we are measuring
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if not tokens:
        return []

    chunks = []
    start  = 0

    while start < len(tokens):
        end        = min(start + chunk_size, len(tokens))
        chunk_text = tokenizer.decode(tokens[start:end], skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)
        start += (chunk_size - overlap)

    return chunks


# Verify on one paper before scaling
test_paper     = papers[0]
full_text      = test_paper.get("full_text", {})
paragraph_lists = full_text.get("paragraphs", [])

# V0: flatten ALL paragraphs into one string — structurally blind
parts = []
for section_paragraphs in paragraph_lists:
    if isinstance(section_paragraphs, list):
        for para in section_paragraphs:
            if isinstance(para, str) and para.strip():
                parts.append(para.strip())

paper_text = " ".join(parts)
test_chunks = chunk_fixed_window(paper_text, tokenizer, CHUNK_SIZE, CHUNK_OVERLAP)

# Check token lengths
lengths = [
    len(tokenizer.encode(c, add_special_tokens=False))
    for c in test_chunks
]

print(f"Paper text length:  {len(paper_text)} chars")
print(f"Chunks produced:    {len(test_chunks)}")
print(f"Token lengths:      min={min(lengths)}  max={max(lengths)}  mean={sum(lengths)/len(lengths):.0f}")
print()
print("First chunk:")
print(f"  '{test_chunks[0][:200]}'")
print()
print("Second chunk (overlap visible):")
print(f"  '{test_chunks[1][:200]}'")

Token indices sequence length is longer than the specified maximum sequence length for this model (4022 > 256). Running this sequence through the model will result in indexing errors


Paper text length:  15750 chars
Chunks produced:    18
Token lengths:      min=214  max=258  mean=250

First chunk:
  'affective events bibref0 are events that typically affect people in positive or negative ways. for example, getting money and playing sports are usually positive to the experiencers ; catching cold an'

Second chunk (overlap visible):
  '##con and a large raw corpus. as illustrated in figure figref1, our key idea is that we can exploit discourse relations bibref4 to efficiently propagate polarity from seed predicates that directly rep'


In [38]:
print("Building V0 chunks across all 888 papers...")
print("(Flattening all paragraphs — structurally blind)")
print()

t0         = time.time()
all_chunks = []
chunk_id   = 0
skipped    = 0

for paper in papers:
    paper_id        = paper.get("id", "unknown")
    full_text       = paper.get("full_text", {})
    paragraph_lists = full_text.get("paragraphs", []) if isinstance(full_text, dict) else []

    # V0: flatten everything — no structural awareness
    parts = []
    for section_paragraphs in paragraph_lists:
        if isinstance(section_paragraphs, list):
            for para in section_paragraphs:
                if isinstance(para, str) and para.strip():
                    parts.append(para.strip())

    paper_text = " ".join(parts)
    if not paper_text.strip():
        skipped += 1
        continue

    for i, chunk_text in enumerate(chunk_fixed_window(paper_text, tokenizer, CHUNK_SIZE, CHUNK_OVERLAP)):
        all_chunks.append({
            "chunk_id":    chunk_id,
            "paper_id":    paper_id,
            "chunk_text":  chunk_text,
            "chunk_index": i,
        })
        chunk_id += 1

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s")
print(f"Total chunks:          {len(all_chunks):,}")
print(f"Papers skipped:        {skipped}")
print(f"Avg chunks per paper:  {len(all_chunks)/888:.0f}")

Building V0 chunks across all 888 papers...
(Flattening all paragraphs — structurally blind)

Done in 11.9s
Total chunks:          22,165
Papers skipped:        1
Avg chunks per paper:  25


In [40]:
## confirm the metadata
print("Sample chunk:")
print(json.dumps(all_chunks[0], indent=2))
print()
print("Sample chunk from middle:")
print(json.dumps(all_chunks[500], indent=2))

Sample chunk:
{
  "chunk_id": 0,
  "paper_id": "1909.00694",
  "chunk_text": "affective events bibref0 are events that typically affect people in positive or negative ways. for example, getting money and playing sports are usually positive to the experiencers ; catching cold and losing one ' s wallet are negative. understanding affective events is important to various natural language processing ( nlp ) applications such as dialogue systems bibref1, question - answering systems bibref2, and humor recognition bibref3. in this paper, we work on recognizing the polarity of an affective event that is represented by a score ranging from $ - 1 $ ( negative ) to 1 ( positive ). learning affective events is challenging because, as the examples above suggest, the polarity of an event is not necessarily predictable from its constituent words. combined with the unbounded combinatorial nature of language, the non - compositionality of affective polarity entails the need for large amounts of world 

In [42]:
with open(CHUNKS_FILE, "w") as f:
    json.dump(all_chunks, f)

size_mb = os.path.getsize(CHUNKS_FILE) / (1024 * 1024)
print(f"Saved {len(all_chunks):,} chunks → {CHUNKS_FILE} ({size_mb:.1f} MB)")

Saved 22,165 chunks → data/v0_chunks.json (25.7 MB)


### Embed all chunks.
We embed in batches to avoid memory spikes. Each batch of 256 chunks gets encoded, normalised, and appended. The L2 normalisation is what makes dot product equal cosine similarity in FAISS.

In [45]:
EMBED_BATCH = 256

print(f"Embedding {len(all_chunks):,} chunks...")
print(f"Batch size: {EMBED_BATCH}")
print()

texts = [c["chunk_text"] for c in all_chunks]
n     = len(texts)
t0    = time.time()

embedding_parts = []

for i in range(0, n, EMBED_BATCH):
    batch = texts[i : i + EMBED_BATCH]
    vecs  = embed_model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,  # L2 normalise → cosine similarity
        show_progress_bar=False,
    )
    embedding_parts.append(vecs)

    # Progress every 5000 chunks
    done    = min(i + EMBED_BATCH, n)
    elapsed = time.time() - t0
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = (n - done) / rate if rate > 0 else 0
    if done % 5000 < EMBED_BATCH or done == n:
        print(f"  [{done:,}/{n:,}]  {elapsed:.0f}s elapsed  {rate:.0f} chunks/s  ETA {eta:.0f}s")

embeddings = np.vstack(embedding_parts).astype("float32")
print()
print(f"Embedding complete.")
print(f"Matrix shape: {embeddings.shape}")
print(f"Total time:   {time.time()-t0:.1f}s")

Embedding 22,165 chunks...
Batch size: 256

  [5,120/22,165]  50s elapsed  102 chunks/s  ETA 167s
  [10,240/22,165]  100s elapsed  102 chunks/s  ETA 116s
  [15,104/22,165]  147s elapsed  102 chunks/s  ETA 69s
  [20,224/22,165]  195s elapsed  104 chunks/s  ETA 19s
  [22,165/22,165]  213s elapsed  104 chunks/s  ETA 0s

Embedding complete.
Matrix shape: (22165, 384)
Total time:   213.1s


In [47]:
print("Building FAISS IndexFlatL2...")

dim   = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"Index built.")
print(f"  Vectors:   {index.ntotal:,}")
print(f"  Dimension: {dim}")

# Save to disk
faiss.write_index(index, INDEX_FILE)
size_mb = os.path.getsize(INDEX_FILE) / (1024 * 1024)
print(f"  Saved → {INDEX_FILE} ({size_mb:.1f} MB)")

Building FAISS IndexFlatL2...
Index built.
  Vectors:   22,165
  Dimension: 384
  Saved → data/v0_index.faiss (32.5 MB)


In [51]:
def retrieve(query: str, model, index, chunks: list, top_k: int) -> list:
    """
    Embed query, search FAISS, return top-K chunk texts.
    V0: pure cosine similarity, no structural filtering.
    """
    q_vec = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    distances, indices = index.search(q_vec, top_k)
    return [chunks[idx]["chunk_text"] for idx in indices[0] if idx != -1]


def generate(query: str, context_chunks: list) -> str:
    """
    Generate answer via Ollama.
    Context chunks concatenated and injected into user message.
    Temperature=0 for reproducibility.
    """
    context = "\n\n---\n\n".join(context_chunks)
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"},
        ],
        options={"temperature": TEMPERATURE},
    )
    return response["message"]["content"].strip()


# Quick smoke test — one query end to end
print("Smoke test — one query through retrieve + generate...")
print()

test_q = queries[0]["question"]
print(f"Question: {test_q}")
print()

t0        = time.time()
retrieved = retrieve(test_q, embed_model, index, all_chunks, TOP_K)
t_retr    = time.time() - t0
print(f"Retrieved {len(retrieved)} chunks in {t_retr*1000:.1f}ms")
print(f"First chunk (first 150 chars): '{retrieved[0][:150]}'")
print()

t0      = time.time()
answer  = generate(test_q, retrieved)
t_gen   = time.time() - t0
print(f"Generated answer in {t_gen:.1f}s:")
print(f"  '{answer[:200]}'")

Smoke test — one query through retrieve + generate...

Question: Why is this work different from text-only UNMT?

Retrieved 5 chunks in 451.2ms
First chunk (first 150 chars): 'not attend over the document given the visual output of the previous layer ( ( display _ form27 ) ), and the document is instead represented through s'

Generated answer in 11.4s:
  'I cannot answer this question from the provided context. The context only discusses the details of a specific NMT experiment and does not mention UNMT (Universal Neural Machine Translation) or its dif'


### RAGAS evaluation function.
RAGAS defaults to GPT-4o-mini as its judge LLM when no LLM is specified. We need to tell it explicitly to use Llama 3.2 3B via Ollama instead
#### wrapping Ollama in a LangChain-compatible interface that RAGAS accepts

In [59]:
!pip install langchain-ollama langchain-community langchain

In [65]:
from langchain_ollama import ChatOllama
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings

# Wrap Llama 3.2 3B as the RAGAS judge LLM
ragas_llm = LangchainLLMWrapper(
    ChatOllama(model=LLM_MODEL, temperature=TEMPERATURE)
)

# Wrap the same embedding model for RAGAS
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBED_MODEL)
)

print("RAGAS LLM and embeddings configured.")
print(f"  Judge LLM:   {LLM_MODEL} via Ollama")
print(f"  Embeddings:  {EMBED_MODEL}")

/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_27416/874755460.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAGAS LLM and embeddings configured.
  Judge LLM:   llama3.2:3b via Ollama
  Embeddings:  sentence-transformers/all-MiniLM-L6-v2


/var/folders/wn/m7ypjd8j3y5fc5flk1tx2hl40000gn/T/ipykernel_27416/874755460.py:12: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [74]:
import numpy as np

def evaluate_query(
    question:     str,
    answer:       str,
    ground_truth: str,
    retrieved:    list,
) -> dict:
    ragas_data = Dataset.from_dict({
        "question":      [question],
        "answer":        [answer],
        "contexts":      [retrieved],
        "ground_truth":  [ground_truth],
        "ground_truths": [[ground_truth]],
    })

    result = evaluate(
        ragas_data,
        metrics=[context_recall, context_precision, faithfulness],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,   # don't crash on JSON parse failures
    )

    def safe_score(key):
        val = result[key]
        # RAGAS returns a list of per-row scores
        score = val[0] if isinstance(val, list) else val
        # Return 0.0 if NaN (parse failure)
        return 0.0 if (score is None or (isinstance(score, float) and np.isnan(score))) else float(score)

    return {
        "context_recall":    safe_score("context_recall"),
        "context_precision": safe_score("context_precision"),
        "faithfulness":      safe_score("faithfulness"),
    }

print("evaluate_query updated — exception handling added.")

evaluate_query updated — exception handling added.


In [76]:
print("Testing RAGAS evaluation with local Ollama judge...")
print()
print(f"Question:     {test_q}")
print(f"Ground truth: {test_ground_truth[:100]}")
print()

t0      = time.time()
metrics = evaluate_query(test_q, answer, test_ground_truth, retrieved)
t_eval  = time.time() - t0

print(f"RAGAS evaluation complete in {t_eval:.1f}s")
print()
print(f"  context_recall:    {metrics['context_recall']:.3f}")
print(f"  context_precision: {metrics['context_precision']:.3f}")
print(f"  faithfulness:      {metrics['faithfulness']:.3f}")

Testing RAGAS evaluation with local Ollama judge...

Question:     Why is this work different from text-only UNMT?
Ground truth: the image can play the role of a pivot “language" to bridge the two languages without paralleled cor



Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[1]: OutputParserException(Invalid json output: The provided context was not useful in arriving at the given answer. The context discusses various corpora and their characteristics, but does not provide any information about how the image can play a role in bridging languages without paralleled corpus.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


RAGAS evaluation complete in 69.8s

  context_recall:    0.000
  context_precision: 0.000
  faithfulness:      0.333


In [78]:
print(f"Running V0 pilot — {N_PILOT} queries")
print("-" * 55)

results     = []
t_run_start = time.time()

for i, q in enumerate(queries):
    t_q = time.time()
    try:
        # Retrieval
        t0        = time.time()
        retrieved = retrieve(q["question"], embed_model, index, all_chunks, TOP_K)
        t_retr    = time.time() - t0

        # Generation
        t0        = time.time()
        generated = generate(q["question"], retrieved)
        t_gen     = time.time() - t0

        # Evaluation
        t0      = time.time()
        metrics = evaluate_query(q["question"], generated, q["answer"], retrieved)
        t_eval  = time.time() - t0

        t_total = time.time() - t_q

        results.append({
            "query_index":       i,
            "paper_id":          q["paper_id"],
            "question":          q["question"],
            "ground_truth":      q["answer"],
            "answer_type":       q["answer_type"],
            "generated_answer":  generated,
            "retrieved_chunks":  retrieved,
            "context_recall":    metrics["context_recall"],
            "context_precision": metrics["context_precision"],
            "faithfulness":      metrics["faithfulness"],
            "timing": {
                "retrieval_s":  round(t_retr, 3),
                "generation_s": round(t_gen, 3),
                "evaluation_s": round(t_eval, 3),
                "total_s":      round(t_total, 3),
            },
            "status": "success",
        })

        print(
            f"[{i+1:2}/{N_PILOT}] {q['answer_type']:12} "
            f"recall={metrics['context_recall']:.3f}  "
            f"precision={metrics['context_precision']:.3f}  "
            f"faith={metrics['faithfulness']:.3f}  "
            f"({t_total:.0f}s)"
        )

    except Exception as e:
        print(f"[{i+1:2}/{N_PILOT}] ERROR: {e}")
        traceback.print_exc()
        results.append({
            "query_index": i,
            "status":      "error",
            "error":       str(e),
            **q
        })

t_run_total = time.time() - t_run_start
print("-" * 55)
print(f"Total time: {t_run_total:.0f}s  ({t_run_total/N_PILOT:.0f}s per query)")

Running V0 pilot — 10 queries
-------------------------------------------------------


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[1]: OutputParserException(Invalid json output: The provided context was not useful in arriving at the given answer. The context discusses various corpora and their characteristics, but does not provide any information about how the image can play a role in bridging languages without paralleled corpus.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )


[ 1/10] extractive   recall=0.000  precision=0.000  faith=0.333  (79s)


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

...

Exception raised in Job[2]: OutputParserException(Invalid json output: Given a question and an answer, analyze the complexity of each sentence in the answer. Break down each sentence into one or more fully understandable statements. Ensure that no pronouns are used in any statement. Format the outputs in JSON.
Please return the output in a JSON format that complies with the following schema as specified in JSON Schema:
{"properties": {"statements": {"description": "The generated statements", "items": {"type": "string"}, "title": "Statements", "type": "array"}}, "required": ["statements"], "title": "StatementGeneratorOutput", "type": "object"}

--------EXAMPLES-----------
Example 1
Input: {
    "question": "Who was Albert Einstein and what is he best known for?",
    "answer": "He was a German-born theoretical physicist, widely acknowledged to be one of the greatest and most influential physicists of all time. He was best known for developing the theory of relativity, he also made impor

[ 9/10] abstractive  recall=0.667  precision=1.000  faith=0.000  (105s)


Evaluating:   0%|          | 0/3 [00:00<?, ?it/s]

Exception raised in Job[2]: OutputParserException(Invalid json output: def analyze_complexity(answer):
    statements = []
    for sentence in answer.split('.'): 
        if sentence.strip():
            words = sentence.split()
            formatted_sentence = ''
            for word in words:
                if word.isupper() and not word.isdigit():
                    formatted_sentence += f'"{word}" ' 
                elif word.isdigit():
                    formatted_sentence += str(word) + ' '
                else:
                    formatted_sentence += word + ' '
            statements.append(formatted_sentence.strip())
    return {
        "statements": statements
    }

input_data = {
    "question": "", 
    "answer": ""
"The text mentions the following languages:\\n\\n1. Indo-European languages:\\n\\t* English\\n\\t* French\\n\\t* German\\n\\t* Spanish\\n\\t* Italian\\n\\t* Portuguese\\n\\t* Swedish\\n\\t* Russian\\n\\t* Polish\\n\\t* Norwegian\\n\\t* Romanian\\n\\t* Dutc

[10/10] extractive   recall=1.000  precision=0.200  faith=0.000  (1431s)
-------------------------------------------------------
Total time: 2955s  (296s per query)


In [82]:
successful = [r for r in results if r.get("status") == "success"]
failed     = [r for r in results if r.get("status") == "error"]

def mean(key, rows):
    vals = [r[key] for r in rows if key in r]
    return round(sum(vals)/len(vals), 4) if vals else None

def mean_timing(key, rows):
    vals = [r["timing"][key] for r in rows if "timing" in r]
    return round(sum(vals)/len(vals), 1) if vals else None

summary = {
    "variant":                "V0 — Naive RAG Baseline",
    "total_queries":          len(results),
    "successful":             len(successful),
    "failed":                 len(failed),
    "mean_context_recall":    mean("context_recall", successful),
    "mean_context_precision": mean("context_precision", successful),
    "mean_faithfulness":      mean("faithfulness", successful),
    "mean_total_s":           mean_timing("total_s", successful),
}

output = {
    "summary": summary,
    "config": {
        "variant":       "V0",
        "embed_model":   EMBED_MODEL,
        "llm_model":     LLM_MODEL,
        "top_k":         TOP_K,
        "chunk_size":    CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "temperature":   TEMPERATURE,
        "n_queries":     len(queries),
    },
    "results": results,
}

with open(RESULTS_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved → {RESULTS_FILE}")

Saved → data/v0_results.json


In [84]:
# Cell 18 — Summary table
print("=" * 55)
print("V0 NAIVE RAG BASELINE — PILOT RESULTS")
print("=" * 55)
print(f"  Queries:           {len(successful)}/{len(results)} successful")
print()
print(f"  Context Recall:    {mean('context_recall', successful)}   (failure < 0.70)")
print(f"  Context Precision: {mean('context_precision', successful)}   (failure < 0.70)")
print(f"  Faithfulness:      {mean('faithfulness', successful)}   (failure < 0.75)")
print()

# Per type
by_type = {}
for r in successful:
    t = r["answer_type"]
    if t not in by_type:
        by_type[t] = []
    by_type[t].append(r)

print("  By answer type:")
for t, rows in sorted(by_type.items()):
    print(f"    {t:12} (n={len(rows)}):  "
          f"recall={mean('context_recall', rows)}  "
          f"precision={mean('context_precision', rows)}  "
          f"faith={mean('faithfulness', rows)}")
print()
print(f"  Mean time/query:   296s")
print(f"  150q estimate:     ~12.3 hours (HPC required)")
print("=" * 55)

V0 NAIVE RAG BASELINE — PILOT RESULTS
  Queries:           10/10 successful

  Context Recall:    0.475   (failure < 0.70)
  Context Precision: 0.27   (failure < 0.70)
  Faithfulness:      0.0833   (failure < 0.75)

  By answer type:
    abstractive  (n=3):  recall=0.4722  precision=0.3333  faith=0.0
    extractive   (n=5):  recall=0.6167  precision=0.14  faith=0.0667
    yes_no       (n=2):  recall=0.125  precision=0.5  faith=0.25

  Mean time/query:   296s
  150q estimate:     ~12.3 hours (HPC required)
